# 02 — Control Mapping Analysis

Analyse the relationship between controls and remediation items.

**Learning notes:**
- A control with zero RI coverage is a *gap* — it exists on paper but nothing is being done to address it.
- A control with very high RI coverage could indicate an over-burdened area, OR that the control domain is generating lots of work.
- Coverage % ≠ effectiveness. This notebook measures mapping coverage only — whether the work is actually fixing the problem is a different question.

**Key questions:**
- Which controls have no RI coverage?
- Which controls are over-represented?
- Which control domains have the weakest coverage?

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.helpers import load_data, set_plot_style, save_processed

set_plot_style()

data = load_data("../data/raw")
incidents = data["incidents"]
ris       = data["ris"]
controls  = data["controls"]
mappings  = data["mappings"]

print(f"Loaded: {len(controls)} controls, {len(mappings)} mappings")

## 1. RI Count per Control

Join mappings onto controls. Controls with no mapping entry will show `ri_count = 0`.

In [ ]:
control_ri_counts = (
    mappings.groupby("control_id")["ri_id"]
    .nunique()
    .reset_index(name="ri_count")
)

controls_coverage = controls.merge(control_ri_counts, on="control_id", how="left")
controls_coverage["ri_count"] = controls_coverage["ri_count"].fillna(0).astype(int)
controls_coverage["has_coverage"] = controls_coverage["ri_count"] > 0

print(f"Controls with coverage    : {controls_coverage['has_coverage'].sum()}")
print(f"Controls without coverage : {(~controls_coverage['has_coverage']).sum()}")

## 2. Zero-Coverage Controls (Gaps)

These controls exist but have no remediation items mapped to them — a compliance and risk finding.

In [ ]:
zero_coverage = controls_coverage[controls_coverage["ri_count"] == 0]
print(f"{len(zero_coverage)} controls have NO RI coverage:")
display(zero_coverage[["control_id", "control_name", "control_domain"]])

## 3. Over-Represented Controls

Controls with more than 2× the average RI count may indicate systemic issues in their domain, or that they are being over-applied.

In [ ]:
mean_ri = controls_coverage["ri_count"].mean()
overrep = (
    controls_coverage[controls_coverage["ri_count"] > mean_ri * 2]
    .sort_values("ri_count", ascending=False)
)
print(f"Average RI coverage per control: {mean_ri:.1f}")
print(f"Controls with >2× average ({mean_ri * 2:.1f}+):")
display(overrep[["control_id", "control_name", "control_domain", "ri_count"]])

## 4. RI Coverage per Control (Chart)

Red bars = zero coverage controls. These are the gaps to address.

In [ ]:
plot_data = controls_coverage.sort_values("ri_count", ascending=True)
colors = ["#d32f2f" if v == 0 else "#1976d2" for v in plot_data["ri_count"]]

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(plot_data["control_name"], plot_data["ri_count"], color=colors)
ax.set_title("RI Coverage per Control  (red = zero coverage)")
ax.set_xlabel("Number of RIs Mapped")
plt.tight_layout()
plt.savefig("../data/processed/chart_control_coverage.png", dpi=120)
plt.show()

## 5. Domain-Level Coverage Summary

Grouping by domain shows which areas of the control framework have the most gaps.

In [ ]:
domain_summary = (
    controls_coverage
    .groupby("control_domain")
    .agg(
        total_controls=("control_id", "count"),
        covered_controls=("has_coverage", "sum"),
        total_ri_mappings=("ri_count", "sum"),
    )
)
domain_summary["coverage_pct"] = (
    domain_summary["covered_controls"] / domain_summary["total_controls"] * 100
).round(1)

display(domain_summary.sort_values("coverage_pct"))

## 6. Domain Coverage Heatmap

Green = fully covered, Red = no coverage. This makes gaps immediately visible.

In [ ]:
heatmap_data = domain_summary[["coverage_pct"]].sort_values("coverage_pct")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    vmin=0, vmax=100,
    ax=ax,
    cbar_kws={"label": "% Controls with RI Coverage"},
)
ax.set_title("Control Domain Coverage %")
ax.set_ylabel("Control Domain")
plt.tight_layout()
plt.savefig("../data/processed/chart_domain_coverage_heatmap.png", dpi=120)
plt.show()

## 7. Save Control Coverage

In [ ]:
path = save_processed(controls_coverage, "control_coverage.csv", "../data/processed")
print(f"Saved: {path}")

## Key Findings

*(Fill in after running)*

- **Zero-coverage controls:** 
- **Weakest domain:** 
- **Most over-represented control:** 
- **Recommendation:** Review zero-coverage controls — either create RIs to address them, or formally accept/retire the control.

---
**Next:** Run `03_team_risk.ipynb` to score teams by risk exposure.